In [2]:
import pandas as pd


def limpiar_inconsistencias_semana6(df):
    """
    Limpieza de inconsistencias de texto, fechas y duplicados (Semana 6),
    sobre un DataFrame ya tratado en nulos (Semana 4).

    Trata:
      - ciudad_tienda, categoria_producto, canal_compra, metodo_pago:
        mojibake condicional, normalización de mayúsculas/espacios,
        consolidación a valores canónicos
      - fecha_pedido: reemplazo de meses en español, parseo de formatos
        mixtos (ISO / D-M-Y), fechas inválidas -> NaT
      - duplicados exactos de fila: eliminados (keep="first")

    Devuelve el DataFrame tratado (no modifica el original in-place).
    """
    df = df.copy()
    cols_texto = ["ciudad_tienda", "categoria_producto", "canal_compra", "metodo_pago"]

    # --- Mojibake condicional (solo repara si detecta el patrón "Ã") ---
    def reparar_mojibake(serie):
        tiene_mojibake = serie.str.contains("Ã", na=False)
        reparada = serie.copy()
        reparada[tiene_mojibake] = (
            serie[tiene_mojibake]
            .str.encode("cp1252", errors="ignore")
            .str.decode("utf-8", errors="ignore")
        )
        return reparada

    for col in cols_texto:
        df[col] = reparar_mojibake(df[col])

    # --- Normalización de formato (espacios, mayúsculas/minúsculas) ---
    for col in cols_texto:
        df[col] = df[col].str.strip().str.title()

    # --- Consolidación a valores canónicos ---
    correcciones_ciudad = {
        "Bog": "Bogotá",
        "Bogot": "Bogotá",              # caso corrupto (encoding)
        "Bogota": "Bogotá",
        "Bogota D.C.": "Bogotá",
        "Bogota Dc": "Bogotá",
        "Mde": "Medellín",
        "Medellin": "Medellín",
        "Medelln": "Medellín",          # caso corrupto (encoding)
        "Ctg": "Cartagena",
        "Cartagena De Indias": "Cartagena",
        "Cucuta": "Cúcuta",
        "Baq": "Barranquilla",
        "B/Quilla": "Barranquilla",
        "Bga": "Bucaramanga",
        "B/Manga": "Bucaramanga",
        "Ibague": "Ibagué",
        "Mzl": "Manizales",
        "Santamarta": "Santa Marta",
        "Sta. Marta": "Santa Marta",
        "Smr": "Santa Marta",
        "Santiago De Cali": "Cali",
        "Manizales, Caldas": "Manizales",
        "Medellin, Ant.": "Medellín",
    }

    correcciones_categoria = {
        "Electronica": "Electrónica",
        "Jugueteria": "Juguetería",
        "Juguetera": "Juguetería",      # caso corrupto (encoding)
    }

    df["ciudad_tienda"] = df["ciudad_tienda"].replace(correcciones_ciudad)
    df["categoria_producto"] = df["categoria_producto"].replace(correcciones_categoria)

    # --- fecha_pedido: meses en español -> número ---
    meses_es_a_num = {
        "ene": "01", "feb": "02", "mar": "03", "abr": "04",
        "may": "05", "jun": "06", "jul": "07", "ago": "08",
        "sep": "09", "oct": "10", "nov": "11", "dic": "12",
    }
    for mes_es, mes_num in meses_es_a_num.items():
        df["fecha_pedido"] = df["fecha_pedido"].str.replace(f"-{mes_es}-", f"-{mes_num}-", regex=False)

    # --- fecha_pedido: clasificar formato y convertir explícitamente ---
    patron_iso = df["fecha_pedido"].str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
    patron_slash = df["fecha_pedido"].str.match(r'^\d{1,2}/\d{1,2}/\d{4}$', na=False)

    fecha_final = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')
    fecha_final[patron_iso] = pd.to_datetime(df.loc[patron_iso, "fecha_pedido"], format="%Y-%m-%d", errors="coerce")
    fecha_final[patron_slash] = pd.to_datetime(df.loc[patron_slash, "fecha_pedido"], format="%d/%m/%Y", errors="coerce")
    df["fecha_pedido"] = fecha_final

    # --- Eliminación de duplicados exactos ---
    df = df.drop_duplicates(keep="first")

    return df

In [4]:
df_s04 = pd.read_csv("../../data\S04_PD_Lozano_DatasetImputacionInicial.csv")
df_s06 = limpiar_inconsistencias_semana6(df_s04)

# Criterio Semana 6: duplicados en 0
print("Duplicados exactos:", df_s06.duplicated().sum())  # debe ser 0

# Criterio Semana 6: una sola forma de escritura por valor real
for col in ["ciudad_tienda", "categoria_producto", "canal_compra", "metodo_pago"]:
    print(col, df_s06[col].nunique(), sorted(df_s06[col].dropna().unique()))

<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\diego\AppData\Local\Temp\ipykernel_6776\3381662965.py:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  df_s04 = pd.read_csv("../../data\S04_PD_Lozano_DatasetImputacionInicial.csv")


Duplicados exactos: 0
ciudad_tienda 12 ['Barranquilla', 'Bogotá', 'Bucaramanga', 'Cali', 'Cartagena', 'Cúcuta', 'Ibagué', 'Manizales', 'Medellín', 'Pereira', 'Santa Marta', 'Villavicencio']
categoria_producto 6 ['Belleza', 'Deportes', 'Electrónica', 'Hogar', 'Juguetería', 'Moda']
canal_compra 4 ['App', 'Marketplace', 'Tienda', 'Web']
metodo_pago 4 ['Efectivo', 'Paypal', 'Tarjeta', 'Transferencia']
